<a href="https://colab.research.google.com/github/Xuli2317/BOT_FPO_DATA/blob/main/BOT_FPO_DATA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import requests
import pandas as pd
from time import sleep
from concurrent.futures import ThreadPoolExecutor, as_completed
import time
import os
import threading

# import os
import re
# import requests
# import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urljoin, unquote

* ต้องมี token.txt โหลดมาลงก่อน

In [ ]:
TOKEN = os.environ["TOKEN"]
headers = {
    "Authorization": TOKEN,
    "Accept": "application/json"}

In [ ]:
ROOT_DIR = "SENTIMENT_INDEX"

BOT_DIR = os.path.join(ROOT_DIR, "BOT")
BOT_CODE_DIR = os.path.join(BOT_DIR, "CODE")

FPO_DIR = os.path.join(ROOT_DIR, "FPO")
FPO_CODE_DIR = os.path.join(FPO_DIR, "CODE")

CHECKPOINT_DIR = os.path.join(BOT_CODE_DIR, "checkpoint")
CAT_DIR = os.path.join(BOT_CODE_DIR, "category")
SE_DIR = os.path.join(BOT_CODE_DIR, "series")
OB_DIR = os.path.join(BOT_CODE_DIR, "observations")

for d in [
    ROOT_DIR,
    BOT_DIR,
    BOT_CODE_DIR,
    FPO_DIR,
    FPO_CODE_DIR,
    CHECKPOINT_DIR,
    CAT_DIR,
    SE_DIR,
    OB_DIR
]:
    os.makedirs(d, exist_ok=True)

# BOT

## category_list

In [ ]:
CATEGORY_CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR,"category_checkpoint.parquet")

In [ ]:
url_cat = "https://gateway.api.bot.or.th/categorylist/category_list/"
if os.path.exists(CATEGORY_CHECKPOINT_FILE):
    df_cat = pd.read_parquet(CATEGORY_CHECKPOINT_FILE)

else:
    r = requests.get(
        url_cat,
        headers=headers,
        timeout=30
    )
    r.raise_for_status()

    data_cat = r.json()

    df_cat = pd.DataFrame(data_cat["result"]["category"])

    df_cat = df_cat.rename(columns={"category": "category_code"})

    df_cat.to_parquet(CATEGORY_CHECKPOINT_FILE,index=False)

print("category rows:", len(df_cat))

Rows: 390


In [ ]:
df_cat.to_excel(os.path.join(CAT_DIR, "category_all.xlsx"),index=False)

df_cat["group"] = (df_cat["category_code"].astype(str).str.split("_").str[0])

for g in ["FI", "FM", "EC"]:

    df_cat_out = df_cat[df_cat["group"] == g].copy()

    df_cat_out.drop(columns=["group"],inplace=True)

    df_cat_out.to_excel(os.path.join(CAT_DIR,f"category_{g}.xlsx"),index=False)

df_cat.drop(columns=["group"], inplace=True)
df_cat.head()

## series_list

In [ ]:
SERIES_CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR,"series_checkpoint.parquet")

In [ ]:
url_series = "https://gateway.api.bot.or.th/categorylist/series_list/"

if os.path.exists(SERIES_CHECKPOINT_FILE):

    df_series = pd.read_parquet(SERIES_CHECKPOINT_FILE)

else:

    def fetch_series(cat_code):

        try:

            r = requests.get(
                url_series,
                headers=headers,
                params={"category": cat_code},
                timeout=30
            )

            if r.status_code != 200:

                return None, {
                    "category_code": cat_code,
                    "status": r.status_code,
                    "message": r.text[:500]
                }

            data = r.json()

            series = data["result"].get("series", [])

            if not series:
                return None, None

            df_tmp = pd.DataFrame(series)

            df_tmp["category_code"] = cat_code

            return df_tmp, None

        except Exception as e:

            return None, {
                "category_code": cat_code,
                "status": "Exception",
                "message": str(e)
            }

    all_series = []
    errors = []

    cat_list = df_cat["category_code"].dropna().unique()

    with ThreadPoolExecutor(max_workers=10) as executor:

        futures = {
            executor.submit(fetch_series, cat_code): cat_code
            for cat_code in cat_list
        }

        for future in as_completed(futures):

            cat_code = futures[future]

            df_tmp, err = future.result()

            if df_tmp is not None:
                all_series.append(df_tmp)

            if err is not None:
                errors.append(err)
                print("ERROR series:", cat_code, err["status"])

    if len(all_series) == 0:
        df_series = pd.DataFrame()
    else:
        df_series = pd.concat(
            all_series,
            ignore_index=True
        )

    df_series_error = pd.DataFrame(errors)

    df_series.to_parquet(
        SERIES_CHECKPOINT_FILE,
        index=False
    )
print("Series rows:", len(df_series))

df_series.head()

In [ ]:
df_series.to_excel(os.path.join(SE_DIR,"series_all.xlsx"),index=False)

## observations

In [ ]:
CHECKPOINT_FILE = os.path.join(CHECKPOINT_DIR,"latest_checkpoint.parquet")
if os.path.exists(CHECKPOINT_FILE):
    df_checkpoint = pd.read_parquet(CHECKPOINT_FILE)
    done_codes = set(df_checkpoint["series_code"])
else:
    df_checkpoint = pd.DataFrame()
    done_codes = set()

df_todo = df_series[
    ~df_series["series_code"].isin(done_codes)
].copy()

print("Done:", len(done_codes))
print("Remaining:", len(df_todo))

Done: 1452
Remaining: 30858


In [ ]:
url_obs = "https://gateway.api.bot.or.th/observations/"

all_obs = df_checkpoint.to_dict("records")
errors = []
obs_lock = threading.Lock()

In [ ]:
def fetch_observations(row):
    try:
        obs_start = pd.to_datetime(row["observation_start"],errors="coerce")

        start_period = max(obs_start,pd.Timestamp("2024-01-01")).strftime("%Y-%m-%d")

        end_period = (pd.to_datetime(row["observation_end"])+ pd.offsets.MonthEnd(0)
        ).strftime("%Y-%m-%d")

        params = {
            "series_code": row["series_code"],
            "start_period": start_period,
            "end_period": end_period,
            "sort_by": "asc"
        }

        r = requests.get(
            url_obs,
            headers=headers,
            params=params,
            timeout=30
        )

        if r.status_code != 200:
            return None, {
                "series_code": row["series_code"],
                "status": r.status_code,
                "message": r.text[:500]
            }

        data = r.json()

        if len(data["result"]["series"]) == 0:
            return None, None

        series = data["result"]["series"][0]

        obs = pd.DataFrame(series["observations"])

        if obs.empty:
            return None, None

        obs["series_code"] = row["series_code"]

        obs["value"] = pd.to_numeric(obs["value"],errors="coerce")

        records = obs[["period_start", "value", "series_code"]].to_dict("records")

        return records, None

    except Exception as e:
        return None, {
            "series_code": row["series_code"],
            "status": "Exception",
            "message": str(e)
        }

In [ ]:
def save_obs_checkpoint(records, tag):
    df_ckpt = pd.DataFrame(records)

    if df_ckpt.empty:
        return

    df_ckpt = df_ckpt.drop_duplicates(subset=["series_code", "period_start"],keep="last")

    df_ckpt["value"] = pd.to_numeric(df_ckpt["value"],errors="coerce")

    df_ckpt.to_parquet(CHECKPOINT_FILE,index=False)

    print(f"Checkpoint saved @ {tag}")

In [ ]:
OBS_MAX_WORKERS = 4
OBS_CHECKPOINT_EVERY = 100

if not df_todo.empty:
    with ThreadPoolExecutor(max_workers=OBS_MAX_WORKERS) as executor:
        futures = {
            executor.submit(fetch_observations, row): row["series_code"]
            for _, row in df_todo.iterrows()
        }

        for i, future in enumerate(as_completed(futures), start=1):
            series_code = futures[future]
            records, err = future.result()

            if records is not None:
                with obs_lock:
                    all_obs.extend(records)

            if err is not None:
                errors.append(err)
                print("ERROR observations:", series_code, err["status"])

            if i % OBS_CHECKPOINT_EVERY == 0:
                with obs_lock:
                    save_obs_checkpoint(all_obs, i)

# final checkpoint after the loop finishes
save_obs_checkpoint(all_obs, "final")

df_latest_raw = pd.DataFrame(all_obs)

if not df_latest_raw.empty:
    df_latest_raw = df_latest_raw[
        ["period_start", "value", "series_code"]
    ].copy()

    df_latest_raw = df_latest_raw.drop_duplicates(
        subset=["series_code", "period_start"],
        keep="last"
    )

    df_latest_raw["value"] = pd.to_numeric(
        df_latest_raw["value"],
        errors="coerce"
    )

    print("Final checkpoint saved")
    print("Rows:", len(df_latest_raw))
    print("Series:", df_latest_raw["series_code"].nunique())
else:
    print("No observations found")

In [ ]:
# merge กับ series master
df_series_clean = df_series[
    [
        "category_code",
        "series_code",
        "series_name_th",
        "series_name_eng",
    ]
].drop_duplicates()

df_latest = pd.merge(
    df_latest_raw,
    df_series_clean,
    on="series_code",
    how="left"
)

df_latest = df_latest[
    [
        "period_start",
        "value",
        "category_code",
        "series_code",
        "series_name_th",
        "series_name_eng"
    ]
]

df_latest.head()

OB_DIR = os.path.join(BOT_CODE_DIR, "observations")

df_latest.to_excel(os.path.join(OB_DIR, "observations_all.xlsx"),index=False
                   )
# แยก FI / FM / EC
df_latest["group"] = (
    df_latest["category_code"]
    .astype(str)
    .str.split("_")
    .str[0]
)

for g in ["FI", "FM", "EC"]:

    df_out = (
        df_latest[df_latest["group"] == g]
        .drop(columns=["group"])
        .copy()
    )

    df_out.to_excel(f"{OB_DIR}/observations_{g}.xlsx",index=False)

df_latest.drop(columns=["group"], inplace=True)
df_latest.head()

In [ ]:
# clean observations
df_obs_clean = df_latest[
    ["period_start", "value", "series_code"]
].copy()

# series master
df_series_clean = df_series[
    [
        "series_code",
        "category_code",
        "series_name_th",
        "series_name_eng",
    ]
].copy()

# category master
df_cat_clean = df_cat[
    [
        "category_code",
        "description_th"
    ]
].drop_duplicates()

# merge category description เข้า series master
df_series_clean = pd.merge(
    df_series_clean,
    df_cat_clean,
    on="category_code",
    how="left"
)

# merge observations + master
df_final = pd.merge(
    df_obs_clean,
    df_series_clean,
    on="series_code",
    how="left"
)

# แปลง value เป็นตัวเลข
df_final["value"] = pd.to_numeric(
    df_final["value"],
    errors="coerce"
)

# เรียงคอลัมน์
column_order = [
    "period_start",
    "category_code",
    "description_th",
    "series_code",
    "series_name_th",
    "series_name_eng",
    "value"
]

df_final = df_final[column_order]
df_final.head()

,period_start,category_code,description_th,series_code,series_name_th,series_name_eng,value
0,2024-Q1,EC_EI_035_S2,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง จำแนกต...,EILPIWAGEQ00561,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง เหมือง...,Labour Cost Index (ordinary time wage) Mining ...,87.17
1,2024-Q1,EC_EI_035,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง,EILPIWAGEQ00561,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง เหมือง...,Labour Cost Index (ordinary time wage) Mining ...,87.17
2,2024-Q1,EC_EI_035_S2,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง จำแนกต...,EILPIWAGEQ00561,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง เหมือง...,Labour Cost Index (ordinary time wage) Mining ...,87.17
3,2024-Q1,EC_EI_035,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง,EILPIWAGEQ00561,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง เหมือง...,Labour Cost Index (ordinary time wage) Mining ...,87.17
4,2024-Q2,EC_EI_035_S2,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง จำแนกต...,EILPIWAGEQ00561,ดัชนีค่าจ้างแรงงาน ครอบคลุมเฉพาะค่าจ้าง เหมือง...,Labour Cost Index (ordinary time wage) Mining ...,87.06


In [ ]:
def make_period_col(x):
    x = str(x)

    if "-Q" in x:
        return x

    if len(x) >= 10 and x[4] == "-" and x[7] == "-":
        return x[:7]

    if len(x) >= 7 and x[4] == "-":
        return x[:7]

    return x

In [ ]:
PIVOT_DIR = os.path.join(BOT_CODE_DIR, "pivot")
os.makedirs(PIVOT_DIR, exist_ok=True)

In [ ]:
df_final["period_col"] = df_final["period_start"].apply(make_period_col)

df_final = df_final[
    ~df_final["period_col"].astype(str).str.match(r"^\d{4}$")
].copy()

index_cols = [
    "description_th",
    "series_name_th"
]

for freq_name, pattern in {
    "Monthly": r"^\d{4}-\d{2}$",
    "Quarterly": r"^\d{4}-Q[1-4]$"
}.items():

    df_freq = df_final[
        df_final["period_col"].astype(str).str.match(pattern)
    ].copy()

    pivot_freq = pd.pivot_table(
        df_freq,
        values="value",
        index=index_cols,
        columns="period_col",
        aggfunc="mean",
        fill_value=0
    ).reset_index()

    pivot_freq.to_excel(
        os.path.join(PIVOT_DIR, f"Data_Pivot_{freq_name}.xlsx"),
        index=False
    )

    for group in ["FI", "FM", "EC"]:

        GROUP_DIR = os.path.join(PIVOT_DIR, group, freq_name)
        os.makedirs(GROUP_DIR, exist_ok=True)

        df_group = df_freq[
            df_freq["category_code"]
            .astype(str)
            .str.startswith(group)
        ].copy()

        for cat_code, df_cat_group in df_group.groupby("category_code"):

            pivot_cat = pd.pivot_table(
                df_cat_group,
                values="value",
                index=index_cols,
                columns="period_col",
                aggfunc="mean",
                fill_value=0
            ).reset_index()

            pivot_cat.to_excel(
                os.path.join(GROUP_DIR, f"{cat_code}.xlsx"),
                index=False
            )

# FPO

In [ ]:
BASE_URL = "https://www.fpo.go.th/main/Statistic-Database.aspx"
SAVE_DIR = FPO_DIR
FPO_CHECKPOINT_DIR = os.path.join(FPO_CODE_DIR,"checkpoint")
FPO_CHECKPOINT_FILE = os.path.join(FPO_CHECKPOINT_DIR,"checkpoint.parquet")

os.makedirs(SAVE_DIR, exist_ok=True)
os.makedirs(FPO_CHECKPOINT_DIR, exist_ok=True)

headers = {"User-Agent": "Mozilla/5.0"}
r = requests.get(BASE_URL, headers=headers, timeout=30)
r.raise_for_status()

soup = BeautifulSoup(r.text, "html.parser")

rows = []
current_category = None

In [ ]:
for tr in soup.select("tr"):
    cells = [td.get_text(" ", strip=True) for td in tr.select("td")]

    if not cells:
        continue

    row_text = " ".join(cells)

    if row_text.startswith("หมวด"):
        current_category = row_text
        continue

    code_match = re.search(r"\b[A-Za-z]+_[A-Za-z]?\d+\b|Macro_RSI", row_text)

    if not code_match:
        continue

    data_code = code_match.group(0)

    links = []

    for a in tr.select("a[href]"):
        href = urljoin(BASE_URL, a["href"])

        img_alt = ""
        img = a.find("img")

        if img:
            img_alt = img.get("alt", "").lower()

        link_type = None
        href_lower = href.lower()

        if "xls" in img_alt or ".xls" in href_lower:
            link_type = "xls"
        elif "pdf" in img_alt or ".pdf" in href_lower:
            link_type = "pdf"
        elif "csv" in img_alt or ".csv" in href_lower:
            link_type = "csv"

        if link_type:
            links.append({
                "type": link_type,
                "url": href
            })

    rows.append({
        "category": current_category,
        "data_code": data_code,
        "raw_text": row_text,
        "links": links
    })

In [ ]:
df_links = pd.DataFrame(rows)
df_files = df_links.explode("links").dropna(subset=["links"]).copy()
df_files["file_type"] = df_files["links"].apply(lambda x: x["type"])
df_files["url"] = df_files["links"].apply(lambda x: x["url"])
df_files = df_files.drop(columns=["links"])
df_files["file_key"] = (
    df_files["data_code"].astype(str)
    + "_"
    + df_files["file_type"].astype(str)
)

df_files.to_excel(os.path.join(SAVE_DIR,"fpo_statistic_links.xlsx"),index=False)
df_files.head()

(77, 6)


,category,data_code,raw_text,file_type,url,file_key
0,หมวดเศรษฐกิจมหภาค,Macro_RSI,Macro_RSI ดัชนีความเชื่อมั่นอนาคตเศรษฐกิจภูมิภ...,xls,https://www.fpo.go.th/main/getattachment/6d6ce...,Macro_RSI_xls
0,หมวดเศรษฐกิจมหภาค,Macro_RSI,Macro_RSI ดัชนีความเชื่อมั่นอนาคตเศรษฐกิจภูมิภ...,pdf,https://www.fpo.go.th/main/getattachment/125a3...,Macro_RSI_pdf
1,หมวดการคลังและภาษีอากร,FIT_D101,FIT_D101 ผลการจัดเก็บรายได้รัฐบาล 29/5/2569 ...,xls,https://www.fpo.go.th/main/getattachment/b5a16...,FIT_D101_xls
1,หมวดการคลังและภาษีอากร,FIT_D101,FIT_D101 ผลการจัดเก็บรายได้รัฐบาล 29/5/2569 ...,pdf,https://www.fpo.go.th/main/getattachment/4dd85...,FIT_D101_pdf
2,หมวดการคลังและภาษีอากร,FIT_D104,FIT_D104 โครงสร้างงบประมาณ 30/4/2569 สายทิพย...,xls,https://www.fpo.go.th/main/getattachment/23950...,FIT_D104_xls


In [ ]:
if os.path.exists(FPO_CHECKPOINT_FILE):
    df_checkpoint = pd.read_parquet(FPO_CHECKPOINT_FILE)
    done_files = set(df_checkpoint["file_key"])

else:
    df_checkpoint = pd.DataFrame(columns=[
        "file_key",
        "category",
        "data_code",
        "file_type",
        "url",
        "path"
    ])
    done_files = set()
df_test = df_files.copy()
df_test = df_files[~df_files["file_key"].isin(done_files)].reset_index(drop=True)

print("Done:", len(done_files))
print("Remaining:", len(df_test))

Loaded checkpoint: 61


In [ ]:
FPO_MAX_WORKERS = 5
FPO_CHECKPOINT_EVERY = 10

fpo_lock = threading.Lock()
new_rows = []

In [ ]:
def download_fpo_file(row):
    file_key = row["file_key"]

    try:
        code = row["data_code"]
        file_type = row["file_type"]
        url = row["url"]

        folder = os.path.join(SAVE_DIR, code)
        os.makedirs(folder, exist_ok=True)

        r = requests.get(
            url,
            headers=headers,
            timeout=60
        )
        r.raise_for_status()

        path = os.path.join(
            folder,
            f"{code}.{file_type}"
        )

        with open(path, "wb") as f:
            f.write(r.content)

        return {
            "file_key": file_key,
            "category": row["category"],
            "data_code": code,
            "file_type": file_type,
            "url": url,
            "path": path
        }, None

    except Exception as e:
        return None, (file_key, e)

In [ ]:
def save_fpo_checkpoint():
    if not new_rows:
        return

    global df_checkpoint

    df_checkpoint = pd.concat(
        [df_checkpoint, pd.DataFrame(new_rows)],
        ignore_index=True
    )

    df_checkpoint = df_checkpoint.drop_duplicates(
        subset=["file_key"],
        keep="last"
    )

    df_checkpoint.to_parquet(
        FPO_CHECKPOINT_FILE,
        index=False
    )

In [ ]:
if not df_test.empty:
    with ThreadPoolExecutor(max_workers=FPO_MAX_WORKERS) as executor:
        futures = {
            executor.submit(download_fpo_file, row): row["file_key"]
            for _, row in df_test.iterrows()
        }

        for i, future in enumerate(as_completed(futures), start=1):
            file_key = futures[future]
            new_row, err = future.result()

            if new_row is not None:
                with fpo_lock:
                    new_rows.append(new_row)
                    done_files.add(file_key)

            if err is not None:
                _, e = err
                print(f"ERROR : {file_key}")
                print(e)

            if i % FPO_CHECKPOINT_EVERY == 0:
                with fpo_lock:
                    save_fpo_checkpoint()
                    new_rows.clear()

# final checkpoint after the loop finishes
with fpo_lock:
    save_fpo_checkpoint()
    new_rows.clear()